# **Legal Precedent Search Engine**

A machine learning research tool that identifies potentially relevant court case precedents from a scraped legal dataset. Users can enter a description of a new court case, and the application returns the most similar historical cases that may provide useful precedent for legal analysis or case preparation.

# **Features**

Automated dataset scraping and preprocessing of legal case records

TF-IDF vectorization for converting case text into numerical feature representations

Keyword-based relevance scoring to prioritize legally significant terms

Random Forest ranking/classification option for alternative result sorting and relevance prediction

Natural language query interface that accepts a user's case description and returns the closest matching precedents

# **How It Works**

Legal case data is scraped and cleaned for analysis; in this case it is taken from HuggingFace which is why those blocks contain placeholders.

Case text is transformed into TF-IDF vectors to capture term importance across the dataset.

User input is vectorized using the same TF-IDF model.

Similarity metrics and a Random Forest-based ranking option are applied to identify the most relevant precedents.

The system returns a ranked list of cases that may support the user's legal argument.

# **Tech Stack**

* Python
* scikit-learn
* TF-IDF Vectorizer
* Random Forest Classifier
* Pandas / NumPy
* Web scraping utilities

# **Purpose**

This project demonstrates the application of natural language processing (NLP) and machine learning to legal research. It is designed as a proof-of-concept system that helps users quickly surface potentially relevant precedents from a large corpus of court decisions, reducing the manual effort required for initial case research.

In [ ]:
!pip install -q datasets scikit-learn pandas pyarrow

In [ ]:
# Package installs.
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Set keywords for model to search as well as a result maximum and timeline parameters.
KEYWORDS = ["INSERT PERTINENT KEYWORDS"]
JURISDICTIONS = ["AREA"]
MAX_CASES = 10000
MIN_YEAR = 1970
MAX_YEAR = 2018

In [ ]:
# Dataset manipulation.
from datasets import load_dataset
# Load the dataset hosted on Hugging Face while ensuring only one record is streamed at a time instead of a full download.
from huggingface_hub import login
login(token="YOUR TOKEN")



dataset = load_dataset(
    "DATASET NAME",
    split="train",
    streaming=True,
)
# Define functions to scout whether an entry passes the criteria in the block above.
def passes_filters(record):
  text = (record.get("text") or "").lower()

  if KEYWORDS and not any(kw.lower() in text for kw in KEYWORDS):
    return False

  if JURISDICTIONS:
    court_field = str(record.get("court", "")) + str(record.get("jurisdiction", ""))
    if not any(j.lower() in court_field.lower() for j in JURISDICTIONS):
      return False

  date_str = str(record.get("decision_date", "") or record.get("date", ""))
  match = re.search(r"(1[9]\d{2}|20\d{2})", date_str)
  if match:
    year = int(match.group(1))
    if year < MIN_YEAR or year > MAX_YEAR:
      return False

  return True

relevant_cases = []
scanned = 0

for record in dataset:
  scanned+=1
  if passes_filters(record):
    relevant_cases.append({
        "id": record.get("id"),
        "text": record.get("text", ""),
        "court": record.get("court", record.get("jurisdiction", "unkonwn")),
        "date": record.get("decision_date", record.get("date", "unknown")),
      })
  if len(relevant_cases) >= MAX_CASES:
    break
  if scanned % 10000 == 0:
    print(f"Scanned {scanned} records, kept {len(relevant_cases)} so far...")

print(f"nDone. Scanned {scanned} records, kept {len(relevant_cases)} matching cases.")

In [ ]:
# Instantiate dataframe variables.
df = pd.DataFrame(relevant_cases)
print(df.shape)
df.head()

In [ ]:
# Imports Colab drive and interacts with file system to avoid reloading the entire data set every time the program is run.
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "YOUR CONTENT DRIVE FILE"

os.makedirs(SAVE_DIR, exist_ok=True)

df.to_parque(f"{SAVE_DIR}/filtered_cases.parquet", index=False)

print(f"Saved {len(df)} cases to {SAVE_DIR}/filtered_cases.parquet")

In [ ]:
from sklearn.feature_extraction.text import TfidVectorizer
from sklearn.neighbors import NearestNeighbors

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words="english"
    sublinear_tf=True
)

case_texts = df["text"].fillna("").tolist()

X = vectorizer.fit_transform(case_texts)

nn_index = NearestNeighbors(n_neighbors=10, metric="cosine")

nn_index.fit(x)

print(f"Indexed {X.shape[0]} cases with {X.shape[1]} features.")

In [ ]:
def find_similar_cases(query_text, top_k=10):
  query_vec = vectorizer.transform([query_text])
  distances, indices = nn_index.kneighbors(query_vec, n_neighbors=top_k)

  results = []

  for dist, idx in zip(distances[0], indices[0]):
    row = df.iloc[idx]

    results.append({
        "similarity": round(1 - dist, 4),
        "court": row["court"],
        "date": date["date"],
        "id": id["id"],
        "snippet": row["text"][:400].replace("\n", " ") + "...",
    })

  return pd.DataFrame(results)

new_case_facts = "YOUR SAMPLE SENTENCE"

find_similar_cases(new_case_facts, top_k=10)